In [ ]:
!pip install opencv-python pupil-apriltags
!pip install numpy

In [1]:
import cv2
import numpy as np
import math
import time
from pupil_apriltags import Detector

# ================= CONFIGURATION =================
# 1. TAG SIZE
TAG_SIZE = 0.0508  # 2 inches in meters

# 2. CAMERA INTRINSICS (Fixed for 640x480)
# We force the camera to 640x480 below, so these 
# approximations (center 320,240) will be accurate enough.
CAMERA_PARAMS = [600, 600, 320, 240]

# 3. REPORTING THRESHOLDS
# Only report if values change by at least this much:
MIN_DIST_CHANGE = 0.02   # 2 cm change
MIN_ANGLE_CHANGE = 2.0   # 2 degrees change
REPORT_INTERVAL = 3.0    # Seconds
# =================================================

# Initialize Detector
at_detector = Detector(
    families='tag36h11',
    nthreads=1,
    quad_decimate=1.0,
    quad_sigma=0.0,
    refine_edges=1,
    decode_sharpening=0.25,
    debug=0
)

# Initialize Camera
cap = cv2.VideoCapture(0)

# FORCE 640x480 Resolution (Crucial for uncalibrated params)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

# State Variables for Tracking
last_print_time = 0
last_pose = None  # Will store tuple (distance, yaw, pitch)

print(f"Monitoring for Tag... (Updates every {REPORT_INTERVAL}s if moved)")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        tags = at_detector.detect(
            gray,
            estimate_tag_pose=True,
            camera_params=CAMERA_PARAMS,
            tag_size=TAG_SIZE
        )

        for tag in tags:
            # --- Visuals ---
            corners = tag.corners.astype(int)
            for i in range(4):
                cv2.line(frame, tuple(corners[i]), tuple(corners[(i+1)%4]), (0, 255, 0), 2)

            # --- Calculations ---
            x = tag.pose_t[0][0]
            y = tag.pose_t[1][0]
            z = tag.pose_t[2][0]

            # 1. Distance
            distance = math.sqrt(x**2 + y**2 + z**2)

            # 2. Yaw (Left/Right)
            yaw_deg = math.degrees(math.atan2(x, z))

            # 3. Pitch (Up/Down) - Inverted so + is Up
            pitch_deg = -math.degrees(math.atan2(y, z))

            # --- Logic: Should we print? ---
            current_time = time.time()
            time_elapsed = current_time - last_print_time
            
            should_print = False

            # Initial print
            if last_pose is None:
                should_print = True
            
            # Check Time AND Movement
            elif time_elapsed >= REPORT_INTERVAL:
                # Unpack last values
                last_dist, last_yaw, last_pitch = last_pose
                
                # Check deltas
                delta_dist = abs(distance - last_dist)
                delta_yaw = abs(yaw_deg - last_yaw)
                delta_pitch = abs(pitch_deg - last_pitch)

                if (delta_dist > MIN_DIST_CHANGE or 
                    delta_yaw > MIN_ANGLE_CHANGE or 
                    delta_pitch > MIN_ANGLE_CHANGE):
                    should_print = True

            if should_print:
                print(f"Update: Dist={distance:.3f}m | Yaw={yaw_deg:.1f}° | Pitch={pitch_deg:.1f}°")
                
                # Update state
                last_print_time = current_time
                last_pose = (distance, yaw_deg, pitch_deg)

        # Show video
        cv2.imshow('Tracker (q to quit)', frame)
        if cv2.waitKey(1) == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

Monitoring for Tag... (Updates every 3.0s if moved)
Update: Dist=0.660m | Yaw=16.1° | Pitch=-19.2°
Update: Dist=0.489m | Yaw=8.6° | Pitch=-7.4°
Update: Dist=0.494m | Yaw=6.5° | Pitch=-8.3°
Update: Dist=0.495m | Yaw=4.5° | Pitch=-8.3°
Update: Dist=0.489m | Yaw=0.1° | Pitch=-10.2°
Update: Dist=0.495m | Yaw=-1.9° | Pitch=-10.2°
Update: Dist=0.495m | Yaw=-3.9° | Pitch=-10.3°
Update: Dist=0.503m | Yaw=-1.9° | Pitch=-10.4°
Update: Dist=0.506m | Yaw=-1.8° | Pitch=-8.3°
Update: Dist=0.254m | Yaw=-2.4° | Pitch=-10.4°
Update: Dist=0.695m | Yaw=6.7° | Pitch=-5.6°
Update: Dist=0.930m | Yaw=9.8° | Pitch=-4.6°
Update: Dist=1.032m | Yaw=9.4° | Pitch=-3.6°


KeyboardInterrupt: 

In [2]:
import cv2
import numpy as np
import math
import time
from pupil_apriltags import Detector

# ================= GLOBALS =================
# Stores the latest target data accessible by other cells/functions
TARGET_STATE = {
    "has_target": False,
    "tag_id": None,
    "distance": 0.0,
    "yaw": 0.0,
    "pitch": 0.0,
    "last_seen_time": 0
}

# Internal tracking for the "3-second print rule"
_LAST_PRINT_STATE = {
    "time": 0,
    "distance": 0.0,
    "yaw": 0.0,
    "pitch": 0.0
}

# Configuration
TAG_SIZE = 0.0508  # 2 inches
CAMERA_PARAMS = [600, 600, 320, 240] # Adjusted for 640x480
PRINT_INTERVAL = 3.0
MIN_CHANGE = 0.02 # 2cm or 2 degrees

def process_frame(frame, detector):
    """
    Takes a video frame, detects tags, updates the global TARGET_STATE,
    and prints to stdout if the 'significant change' criteria are met.
    """
    global TARGET_STATE, _LAST_PRINT_STATE
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    tags = detector.detect(
        gray,
        estimate_tag_pose=True,
        camera_params=CAMERA_PARAMS,
        tag_size=TAG_SIZE
    )

    # Reset target flag if no tags found
    if not tags:
        TARGET_STATE["has_target"] = False
        return

    # Process the first tag found
    tag = tags[0]
    
    # 1. Calculate Pose
    x = tag.pose_t[0][0]
    y = tag.pose_t[1][0]
    z = tag.pose_t[2][0]

    distance = math.sqrt(x**2 + y**2 + z**2)
    yaw = math.degrees(math.atan2(x, z))
    pitch = -math.degrees(math.atan2(y, z))

    # 2. Update Global State
    TARGET_STATE.update({
        "has_target": True,
        "tag_id": tag.tag_id,
        "distance": distance,
        "yaw": yaw,
        "pitch": pitch,
        "last_seen_time": time.time()
    })

    # 3. Handle Visuals (Draw on frame)
    corners = tag.corners.astype(int)
    for i in range(4):
        cv2.line(frame, tuple(corners[i]), tuple(corners[(i+1)%4]), (0, 255, 0), 2)
    cv2.putText(frame, f"ID:{tag.tag_id}", tuple(corners[0]), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # 4. Handle Stdout Printing (The 3-second rule)
    current_time = time.time()
    time_elapsed = current_time - _LAST_PRINT_STATE["time"]

    # Check for significant change
    delta_dist = abs(distance - _LAST_PRINT_STATE["distance"])
    delta_yaw = abs(yaw - _LAST_PRINT_STATE["yaw"])
    delta_pitch = abs(pitch - _LAST_PRINT_STATE["pitch"])

    is_significant = (delta_dist > MIN_CHANGE or 
                      delta_yaw > MIN_CHANGE or 
                      delta_pitch > MIN_CHANGE)

    # Print if (First Time) OR (Time Elapsed AND Changed)
    if _LAST_PRINT_STATE["time"] == 0 or (time_elapsed > PRINT_INTERVAL and is_significant):
        print(f"Target Acquired: Dist={distance:.3f}m | Yaw={yaw:.1f}° | Pitch={pitch:.1f}°")
        
        # Update print state
        _LAST_PRINT_STATE["time"] = current_time
        _LAST_PRINT_STATE["distance"] = distance
        _LAST_PRINT_STATE["yaw"] = yaw
        _LAST_PRINT_STATE["pitch"] = pitch

In [3]:
# Initialize Hardware
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

at_detector = Detector(families='tag36h11', nthreads=1)

print("Starting Loop. Press 'q' to quit...")

try:
    while True:
        # 1. Capture
        ret, frame = cap.read()
        if not ret:
            break

        # 2. Process (Updates Globals & Prints)
        process_frame(frame, at_detector)

        # 3. Optional: Access Global State here for Missile Logic later
        # e.g., if TARGET_STATE["has_target"]: fire_missile()

        # 4. Display
        cv2.imshow('Monitor', frame)
        if cv2.waitKey(1) == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

Starting Loop. Press 'q' to quit...
Target Acquired: Dist=0.520m | Yaw=-8.0° | Pitch=-15.4°
Target Acquired: Dist=0.464m | Yaw=-14.6° | Pitch=-8.6°
Target Acquired: Dist=0.467m | Yaw=-14.7° | Pitch=-8.5°
Target Acquired: Dist=0.481m | Yaw=-22.9° | Pitch=-4.2°
Target Acquired: Dist=0.792m | Yaw=-18.1° | Pitch=-4.0°
Target Acquired: Dist=1.064m | Yaw=-14.9° | Pitch=3.1°
Target Acquired: Dist=0.814m | Yaw=-15.3° | Pitch=-1.2°
Target Acquired: Dist=0.817m | Yaw=-17.9° | Pitch=0.8°
Target Acquired: Dist=0.826m | Yaw=-17.7° | Pitch=-1.9°
Target Acquired: Dist=0.807m | Yaw=-15.7° | Pitch=0.2°
Target Acquired: Dist=0.812m | Yaw=-16.6° | Pitch=-0.2°


KeyboardInterrupt: 